# Gillon, Lecoq et al. (2023): ROI matching over session orderings

Paper: doi:10.1038/s41597-023-02214-y. Target dataset DANDI:000336 (version 0.260325.1802), which does
not hold the paper's sessions; the data are read from DANDI:000037, version 0.230426.0054, as in
`reproduce_gillon_2023.ipynb`, whose header gives the reasons for both versions. This notebook tests one
claim, fig8-ordering-variation; the other claims are in the main notebook.

The paper matched ROIs across sessions with a custom-modified version of the Allen Institute's n-way
matching package (github.com/AllenInstitute/ophys_nway_matching) that is not public, adding for
dendrites that matches must overlap by at least 10 to 20% (Methods, ROI tracking across sessions). The
claims file sets the test: the public package at a pinned commit (here 9d02ff5abca4580caad0e13bab60355ff91d4b62,
the latest, version 0.6.0) with default parameters, over the six orderings of each analysed mouse's
three sessions, with the overlap threshold at (a) 10% or (b) 20% for dendritic planes, and the
variation over orderings measured as (i) range over mean or (ii) SD over mean.

Inputs to the package per session: the ROI masks of the file (`image_mask` > 0) and the plane's max
projection image (`max_projection`), which stands in for the average projection the package expects
(the files hold no average projection).

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import credassign as ca

pd.set_option("display.width", 200)
T0 = time.time()
mouse_df = ca.read_mouse_df()
analysed = mouse_df[(mouse_df.pass_fail == "P") & mouse_df.sess_n.between(1, 3)].sort_values(["mouseid", "sess_n"])
basic = ca.list_assets()
basic = basic[basic.path.str.endswith("_behavior+ophys.nwb")]
basic["session"] = basic.path.str.extract(r"_ses-(\d+)_")[0]
sessions_by_mouse, rows = {}, []
for mouse, g in analysed.groupby("mouseid"):
    sessions_by_mouse[str(mouse)] = []
    for sid in g.sessid.astype(str):
        f = ca.open_h5(basic.loc[basic.session == sid, "asset_id"].iloc[0])
        masks, img = ca.roi_masks(f)
        tracking = f["processing/ophys/ImageSegmentation/PlaneSegmentation/tracking_id"][:]
        f.close()
        sessions_by_mouse[str(mouse)].append((sid, masks, img))
        rows.append({"mouse": str(mouse), "session": sid, "plane": g.plane.iloc[0], "line": g.line.iloc[0],
                     "rois": len(masks), "empty_masks": int((masks.sum(axis=(1, 2)) == 0).sum()),
                     "mask_px_median": int(np.median(masks.sum(axis=(1, 2)))), "image": img.shape,
                     "image_range": (int(img.min()), int(img.max())), "tracked_in_file": int(np.sum(~np.isnan(tracking)))})
loaded = pd.DataFrame(rows)
print(loaded.to_string(index=False))
print(len(loaded), "sessions from", loaded.mouse.nunique(), "mice; loaded in", round((time.time() - T0) / 60, 1), "min")

## fig8-ordering-variation

> The number of matched ROIs depends on the order in which sessions are aligned, and this variation is
> far smaller for somata than for dendrites.

Somatic planes are matched without an overlap threshold (the Methods add it for dendrites only);
dendritic planes with the 10% and 20% thresholds, and without one as a variant. A matched ROI is a match
across all three sessions.

In [ ]:
t = time.time()
counts = ca.ordering_counts(sessions_by_mouse, thresholds=(0.1, 0.2), workers=4)
print("matching ran in", round((time.time() - t) / 60, 1), "min")
counts = counts.merge(loaded.groupby("mouse").agg(plane=("plane", "first"), line=("line", "first")), on="mouse")
print(counts.to_string(index=False))

In [ ]:
def variation(x):
    x = np.asarray(x, float)
    return pd.Series({"mean": x.mean(), "min": x.min(), "max": x.max(),
                      "i_range_over_mean": (x.max() - x.min()) / x.mean() if x.mean() else np.nan,
                      "ii_sd_over_mean": x.std(ddof=1) / x.mean() if x.mean() else np.nan})

res = []
for mouse, g in counts.groupby("mouse"):
    plane = g.plane.iloc[0]
    cols = ["all"] if plane == "soma" else ["iou>=0.1", "iou>=0.2", "linked iou>=0.1", "linked iou>=0.2", "all"]
    for col in cols:
        res.append({"mouse": mouse, "plane": plane, "line": g.line.iloc[0],
                    "threshold": {"all": "none", "iou>=0.1": "a (10%)", "iou>=0.2": "b (20%)",
                                  "linked iou>=0.1": "a (10%), linked pairs", "linked iou>=0.2": "b (20%), linked pairs"}[col], **variation(g[col])})
res = pd.DataFrame(res)
tracked = loaded.groupby("mouse").tracked_in_file.first()
res["tracked_in_file"] = res.mouse.map(tracked)
print(res.round(4).to_string(index=False))
soma = res[res.plane == "soma"]
summary = []
for thr in ("a (10%)", "b (20%)", "a (10%), linked pairs", "b (20%), linked pairs", "none"):
    dend = res[(res.plane == "dend") & (res.threshold == thr)]
    for measure in ("i_range_over_mean", "ii_sd_over_mean"):
        summary.append({"dendritic threshold": thr, "measure": measure, "soma_mean": soma[measure].mean(),
                        "dend_mean": dend[measure].mean(), "soma_smaller": soma[measure].mean() < dend[measure].mean(),
                        "soma_max": soma[measure].max(), "dend_min": dend[measure].min()})
summary = pd.DataFrame(summary)
print(summary.round(4).to_string(index=False))
print("somatic mice with any variation over orderings:", int((soma.i_range_over_mean > 0).sum()), "of", len(soma))

A three-session match can join two ROIs that have no pairwise match of their own (the package links
them through the third session). The main reading counts such a pair as IoU 0, so the match fails any
threshold; the variant "linked pairs" applies the threshold only to the pairs that have a pairwise
match. Below, per mouse, how many three-session matches have such a pair, and the counts under each
reading, as the mean over the six orderings, next to the tracked count in the files.

In [ ]:
links = counts.groupby("mouse").agg(plane=("plane", "first"), all=("all", "mean"), no_direct_link=("no_direct_link", "mean"),
                                    **{c: (c, "mean") for c in ["iou>=0.1", "iou>=0.2", "linked iou>=0.1", "linked iou>=0.2"]})
links["tracked_in_file"] = links.index.map(tracked)
links["dropped_for_link_only_10"] = links["linked iou>=0.1"] - links["iou>=0.1"]
links["dropped_for_link_only_20"] = links["linked iou>=0.2"] - links["iou>=0.2"]
print(links.round(1).to_string())

### Check against the package's own run

`ordering_counts` combines directed pairwise matchings computed once, in place of running
`NwayMatching.run` for every ordering. As a check, the package's `run` is called directly on all six
orderings of the smallest somatic mouse (418779) and the smallest dendritic mouse (420011), and its
count of three-session matches is set beside the count above.

In [ ]:
import itertools
direct = []
for m in ("418779", "420011"):
    for order in itertools.permutations(range(3)):
        name = "-".join(str(k + 1) for k in order)
        n = ca.direct_run_count(sessions_by_mouse[m], order)
        direct.append({"mouse": m, "order": name, "direct_run": n,
                       "ordering_counts": int(counts.loc[(counts.mouse == m) & (counts.order == name), "all"].iloc[0])})
direct = pd.DataFrame(direct)
direct["equal"] = direct.direct_run == direct.ordering_counts
print(direct.to_string(index=False))
print("orderings with equal counts:", int(direct.equal.sum()), "of", len(direct))
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")

**Result: reproduced.** In all six somatic mice the number of ROIs matched across the three sessions
varies by at most one over the six orderings (range over mean 0 to 0.017, 4 of 6 mice with no
variation). In the five dendritic mice it varies more: with the 10% overlap threshold the range over
mean is 0.036 to 0.26 (mean 0.136, against 0.004 for somata), with 20% 0.053 to 0.41 (mean 0.161), and
the SD over mean gives the same order. Every dendritic mouse varies more than every somatic one under
both thresholds and both measures, and also when the threshold applies only to pairs with a direct
pairwise match (range over mean 0.029 to 0.40, mean 0.113 at 10% and 0.172 at 20%). Without a threshold
the dendritic mean is still larger (range over mean 0.040 against 0.004), though 433451 then varies less
than somatic mouse 408021.

The result rests on the public package and on reading the overlap as intersection over union. With that
reading the dendritic match counts fall well below the tracked counts in the files (for example 20 and
23 on average for 433414 and 433448, against 118 and 112), Xfiles'. Part of the loss comes from three-session matches with a pair of ROIs that has no direct pairwise
match (19 to 307 such matches per dendritic mouse, against 1 to 4 per somatic mouse): applying the
threshold only to the pairs that have one keeps 5 to 28 more matches per dendritic mouse at 10%, still
below the tracked counts (34 and 44 for 433414 and 433448). The IoU threshold, not the linking rule,
removes most matches. Without a threshold the dendritic counts are about 1.4 to 2.6 times the tracked
counts. The package's own run gives the same count as the reconstruction in all 12 orderings checked
(418779 and 420011).

## Summary

fig8-ordering-variation: reproduced, resting on assumptions (the public matching package in place of the
paper's modified one, and the dendritic overlap read as intersection over union). The full summary of
all claims is in `reproduce_gillon_2023.ipynb`.